In [1]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
@dataclass
class ClusterResult:
    
    algorithm: str
    labels: np.ndarray
    metadata: Dict


@dataclass
class EvaluationResult:

    algorithm: str
    silhouette: float
    calinski: float
    davies: float


@dataclass
class PipelineState:

    raw_df: pd.DataFrame
    feature_df: pd.DataFrame
    X_scaled: np.ndarray
    X_embedding: np.ndarray

    cluster_results: Dict[str, ClusterResult] = None
    evaluation_results: Dict[str, EvaluationResult] = None

In [3]:
class DataInputService:

    def __init__(self, file_path: str):
        self.file_path = file_path

    def load(self) -> pd.DataFrame:

        df = pd.read_csv(self.file_path)

        if df.empty:
            raise ValueError("Dataset is empty")

        return df


    def validate(self, df: pd.DataFrame) -> Dict:

        report = {
            "rows": len(df),
            "columns": len(df.columns),
            "missing_values": df.isna().sum().to_dict(),
            "duplicates": int(df.duplicated().sum())
        }

        return report

In [4]:
class FeatureEngineeringPipeline:

    def __init__(self, pca_variance: float = 0.85):

        self.scaler = StandardScaler()
        self.pca = PCA(n_components=pca_variance)


    def run(self, df: pd.DataFrame):

        numeric_df = df.select_dtypes(include=["int64","float64"]).copy()

        X_scaled = self.scaler.fit_transform(numeric_df)

        X_emb = self.pca.fit_transform(X_scaled)

        return numeric_df, X_scaled, X_emb

In [5]:
class BaseClustering:

    def fit(self, X: np.ndarray):
        raise NotImplementedError

In [6]:
class KMeansRunner(BaseClustering):

    def __init__(self, k_min=2, k_max=10):

        self.k_min = k_min
        self.k_max = k_max


    def fit(self, X):

        best_k = None
        best_score = -1
        best_labels = None

        for k in range(self.k_min, self.k_max):

            model = KMeans(n_clusters=k, random_state=42)

            labels = model.fit_predict(X)

            score = silhouette_score(X, labels)

            if score > best_score:

                best_score = score
                best_k = k
                best_labels = labels

        metadata = {
            "best_k": best_k,
            "silhouette": best_score
        }

        return ClusterResult("kmeans", best_labels, metadata)

In [7]:
class DBSCANRunner(BaseClustering):

    def __init__(self, eps=0.5, min_samples=5):

        self.eps = eps
        self.min_samples = min_samples


    def fit(self, X):

        model = DBSCAN(eps=self.eps, min_samples=self.min_samples)

        labels = model.fit_predict(X)

        metadata = {
            "clusters": len(set(labels)) - (1 if -1 in labels else 0)
        }

        return ClusterResult("dbscan", labels, metadata)

In [8]:
class GMMRunner(BaseClustering):

    def __init__(self, k_min=2, k_max=10):

        self.k_min = k_min
        self.k_max = k_max


    def fit(self, X):

        best_k = None
        best_bic = np.inf
        best_labels = None

        for k in range(self.k_min, self.k_max):

            model = GaussianMixture(n_components=k)

            model.fit(X)

            bic = model.bic(X)

            if bic < best_bic:

                best_bic = bic
                best_k = k
                best_labels = model.predict(X)

        metadata = {
            "best_k": best_k,
            "bic": best_bic
        }

        return ClusterResult("gmm", best_labels, metadata)

In [9]:
class ClusteringEngine:

    def __init__(self):

        self.runners = {
            "kmeans": KMeansRunner(),
            "dbscan": DBSCANRunner(),
            "gmm": GMMRunner()
        }


    def run(self, X):

        results = {}

        for name, runner in self.runners.items():

            result = runner.fit(X)

            results[name] = result

        return results

In [10]:
class ClusterEvaluationEngine:

    def evaluate(self, X, cluster_results):

        evaluations = {}

        for name, result in cluster_results.items():

            labels = result.labels

            if len(set(labels)) <= 1:
                continue

            metrics = EvaluationResult(
                algorithm=name,
                silhouette=silhouette_score(X, labels),
                calinski=calinski_harabasz_score(X, labels),
                davies=davies_bouldin_score(X, labels)
            )

            evaluations[name] = metrics

        return evaluations